# Treino local com Llama + Unsloth no Google Colab

Este notebook treina um adapter LoRA/QLoRA para o fluxo multifacetado de triagem em saúde e segurança da mulher, utilizando múltiplos datasets integrados (planejamento familiar, emergências obstétricas, protocolos, oncologia, triagem e detecção de violência).

Antes de rodar: em `Runtime > Change runtime type`, selecione GPU. Uma T4 costuma funcionar para o modelo 8B em 4 bits, com batch pequeno.


## 1. Instalar dependências

A instalação pode levar alguns minutos e talvez o Colab peça para reiniciar o runtime. Se reiniciar, rode as células novamente a partir daqui.


In [1]:
%%capture
!pip install -U unsloth
!pip install -U --no-deps trl peft accelerate bitsandbytes datasets transformers


## 2. Conferir GPU


In [2]:
!nvidia-smi


Sun May 17 02:03:02 2026       
+-----------------------------------------------------------------------------------------+
| NVIDIA-SMI 580.82.07              Driver Version: 580.82.07      CUDA Version: 13.0     |
+-----------------------------------------+------------------------+----------------------+
| GPU  Name                 Persistence-M | Bus-Id          Disp.A | Volatile Uncorr. ECC |
| Fan  Temp   Perf          Pwr:Usage/Cap |           Memory-Usage | GPU-Util  Compute M. |
|                                         |                        |               MIG M. |
|=========================================+========================+======================|
|   0  Tesla T4                       Off |   00000000:00:04.0 Off |                    0 |
| N/A   47C    P8              9W /   70W |       0MiB /  15360MiB |      0%      Default |
|                                         |                        |                  N/A |
+-----------------------------------------+-----

## 3. Montar Google Drive e definir caminhos

O notebook usa o Drive por padrão para ler o dataset e salvar o adapter/tokenizer treinado.


In [25]:
from pathlib import Path
from google.colab import drive
import shutil

drive.mount('/content/drive')


Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).


In [33]:
PROJECT_DIR = Path('/content/drive/MyDrive/techchallenge3')
DATA_DIR = PROJECT_DIR / 'data'
OUTPUT_DIR = PROJECT_DIR / 'outputs' / 'llama-womens-health-triage-lora'

DATA_DIR.mkdir(parents=True, exist_ok=True)
OUTPUT_DIR.parent.mkdir(parents=True, exist_ok=True)

DATASET_FILES = [
    "planejamento_familiar.jsonl",
    "protocolos_atendimento_gine_obs.jsonl",
    "protocolos_triagem_cancer.jsonl",
    "casos_triagem_obstetricia.jsonl",
    "triagem_saude_mulher_dataset.jsonl",
    "violence_detection_finetuning.jsonl"
]

for filename in DATASET_FILES:
    dest_path = DATA_DIR / filename
    local_upload_path = Path('/content') / filename

    if not dest_path.exists() and local_upload_path.exists():
        shutil.copy2(local_upload_path, dest_path)
        print(f'Dataset copiado para o Drive: {dest_path}')

for filename in DATASET_FILES:
    assert (DATA_DIR / filename).exists(), f'Dataset {filename} não encontrado no Drive ou em /content.'

print('Todos os caminhos validados com sucesso!')
print('Saída do modelo:', OUTPUT_DIR)


Todos os caminhos validados com sucesso!
Saída do modelo: /content/drive/MyDrive/techchallenge3/outputs/llama-womens-health-triage-lora


## 4. Carregar e validar o JSONL


In [34]:
import json
from statistics import mean

def load_chat_jsonl(path):
    examples = []
    with open(path, encoding='utf-8') as file:
        for line_number, line in enumerate(file, start=1):
            if not line.strip():
                continue
            payload = json.loads(line)
            messages = payload.get('messages')
            if not isinstance(messages, list) or len(messages) < 3:
                raise ValueError(f'Linha {line_number} sem messages válidas')
            for message in messages:
                if message.get('role') not in {'system', 'user', 'assistant'}:
                    raise ValueError(f'Linha {line_number} com role inválido: {message}')
                if not message.get('content'):
                    raise ValueError(f'Linha {line_number} com content vazio: {message}')
            examples.append(payload)
    if not examples:
        raise ValueError('Dataset vazio')
    return examples


In [35]:
examples = []

for filename in DATASET_FILES:
    file_path = DATA_DIR / filename
    try:
        file_examples = load_chat_jsonl(file_path)
        examples.extend(file_examples)
        print(f'Sucesso: {len(file_examples)} exemplos carregados de {filename}')
    except Exception as e:
        print(f'Erro ao carregar {filename}: {e}')

print(f'\n--> Total de exemplos combinados para o treinamento: {len(examples)}')


Sucesso: 50 exemplos carregados de planejamento_familiar.jsonl
Sucesso: 55 exemplos carregados de protocolos_atendimento_gine_obs.jsonl
Sucesso: 39 exemplos carregados de protocolos_triagem_cancer.jsonl
Sucesso: 36 exemplos carregados de casos_triagem_obstetricia.jsonl
Sucesso: 30 exemplos carregados de triagem_saude_mulher_dataset.jsonl
Sucesso: 60 exemplos carregados de violence_detection_finetuning.jsonl

--> Total de exemplos combinados para o treinamento: 270


In [36]:
len(examples), examples[0]['messages'][1]

(270,
 {'role': 'user',
  'content': 'Mulher de 36 anos, tabagista (15 cigarros/dia), hipertensa controlada com medicação, solicita início de método contraceptivo na consulta de planejamento familiar. Qual a conduta e quais métodos são contraindicados?'})

## 5. Carregar Llama com Unsloth

O modelo abaixo é carregado em 4 bits para reduzir VRAM. Você pode trocar por outro modelo compatível do Unsloth.


In [ ]:
from unsloth import FastLanguageModel

MODEL_NAME = 'unsloth/Meta-Llama-3.1-8B-Instruct-bnb-4bit'
MAX_SEQ_LENGTH = 2048
LOAD_IN_4BIT = True

model, tokenizer = FastLanguageModel.from_pretrained(
    model_name=MODEL_NAME,
    max_seq_length=MAX_SEQ_LENGTH,
    dtype=None,
    load_in_4bit=LOAD_IN_4BIT,
)


🦥 Unsloth: Will patch your computer to enable 2x faster free finetuning.
🦥 Unsloth Zoo will now patch everything to make training faster!
==((====))==  Unsloth 2026.5.2: Fast Llama patching. Transformers: 5.8.1.
   \\   /|    Tesla T4. Num GPUs = 1. Max memory: 14.563 GB. Platform: Linux.
O^O/ \_/ \    Torch: 2.10.0+cu128. CUDA: 7.5. CUDA Toolkit: 12.8. Triton: 3.6.0
\        /    Bfloat16 = FALSE. FA [Xformers = 0.0.35. FA2 = False]
 "-____-"     Free license: http://github.com/unslothai/unsloth
Unsloth: Fast downloading is enabled - ignore downloading bars which are red colored!


model.safetensors:   0%|          | 0.00/5.70G [00:00<?, ?B/s]

Loading weights:   0%|          | 0/291 [00:00<?, ?it/s]

generation_config.json:   0%|          | 0.00/239 [00:00<?, ?B/s]

config.json: 0.00B [00:00, ?B/s]

tokenizer_config.json: 0.00B [00:00, ?B/s]

tokenizer.json:   0%|          | 0.00/17.2M [00:00<?, ?B/s]

special_tokens_map.json:   0%|          | 0.00/454 [00:00<?, ?B/s]

## 6. Formatar com chat template e tokenizar preview

Aqui o dataset passa pelo tokenizer. O objetivo é medir tamanho em tokens e verificar se haverá truncamento.


In [ ]:
def render_messages(messages, tokenizer):
    return tokenizer.apply_chat_template(
        messages,
        tokenize=False,
        add_generation_prompt=False,
    )

def format_dataset_into_model_input(examples, tokenizer):
    return [
        {
            'text': render_messages(example['messages'], tokenizer),
            'metadata': example.get('metadata', {}),
        }
        for example in examples
    ]


In [ ]:
formatted_examples = format_dataset_into_model_input(examples, tokenizer)

token_lengths = []
truncated = 0
for example in formatted_examples:
    tokens = tokenizer(
        example['text'],
        truncation=True,
        max_length=MAX_SEQ_LENGTH,
        add_special_tokens=False,
    )
    length = len(tokens['input_ids'])
    token_lengths.append(length)
    if length >= MAX_SEQ_LENGTH:
        truncated += 1

{
    'examples': len(formatted_examples),
    'min_tokens': min(token_lengths),
    'avg_tokens': round(mean(token_lengths), 2),
    'max_tokens': max(token_lengths),
    'truncated_examples': truncated,
}

In [ ]:
formatted_examples

## 7. Preparar LoRA


In [ ]:
model = FastLanguageModel.get_peft_model(
    model,
    r=16,
    target_modules=[
        'q_proj', 'k_proj', 'v_proj', 'o_proj',
        'gate_proj', 'up_proj', 'down_proj',
    ],
    lora_alpha=16,
    lora_dropout=0.0,
    bias='none',
    use_gradient_checkpointing='unsloth',
    random_state=3407,
    use_rslora=False,
    loftq_config=None,
)


## 8. Treinar

Para demonstração, `max_steps=150`. Para um treino mais completo, aumente esse valor e monitore perda/overfitting.

A saída do treinamento também usa `OUTPUT_DIR`, que já aponta para o Google Drive.


In [ ]:
from datasets import Dataset
from trl import SFTConfig, SFTTrainer
import inspect

train_dataset = Dataset.from_list(formatted_examples)

sft_params = inspect.signature(SFTConfig.__init__).parameters
sft_kwargs = {
    'output_dir': str(OUTPUT_DIR),
    'per_device_train_batch_size': 2,
    'gradient_accumulation_steps': 4,
    'warmup_steps': 5,
    'max_steps': 150,
    'learning_rate': 2e-4,
    'fp16': True,
    'logging_steps': 1,
    'optim': 'adamw_8bit',
    'weight_decay': 0.01,
    'lr_scheduler_type': 'linear',
    'seed': 3407,
    'report_to': 'none',
    'dataset_text_field': 'text',
    'packing': False,
    'padding_free': False
}

if 'max_length' in sft_params:
    sft_kwargs['max_length'] = MAX_SEQ_LENGTH
elif 'max_seq_length' in sft_params:
    sft_kwargs['max_seq_length'] = MAX_SEQ_LENGTH

sft_kwargs = {key: value for key, value in sft_kwargs.items() if key in sft_params}
sft_config = SFTConfig(**sft_kwargs)

trainer_params = inspect.signature(SFTTrainer.__init__).parameters
trainer_kwargs = {
    'model': model,
    'train_dataset': train_dataset,
    'args': sft_config
}

if 'processing_class' in trainer_params:
    trainer_kwargs['processing_class'] = tokenizer
elif 'tokenizer' in trainer_params:
    trainer_kwargs['tokenizer'] = tokenizer

trainer = SFTTrainer(**trainer_kwargs)
trainer.train()


## 9. Salvar adapter e tokenizer no Google Drive

Esta célula salva explicitamente o adapter LoRA e o tokenizer em `MyDrive/techchallenge3/outputs/llama-womens-health-triage-lora`.


In [ ]:
OUTPUT_DIR.mkdir(parents=True, exist_ok=True)
model.save_pretrained(str(OUTPUT_DIR))
tokenizer.save_pretrained(str(OUTPUT_DIR))

saved_files = sorted(path.name for path in OUTPUT_DIR.iterdir())
print('Modelo/tokenizer salvos em:', OUTPUT_DIR)
saved_files


## 10. Testar inferência


In [ ]:
FastLanguageModel.for_inference(model)

messages = [
    {
        'role': 'system',
        'content': 'Você é um assistente de apoio à triagem em saúde e segurança da mulher. Responda somente em JSON válido com classificação de risco, sinais de alerta, próxima etapa e resposta acolhedora.',
    },
    {
        'role': 'user',
        'content': 'Não aconteceu nada demais, somente um empurrão.',
    },
]

inputs = tokenizer.apply_chat_template(
    messages,
    tokenize=True,
    add_generation_prompt=True,
    return_tensors='pt',
).to('cuda')

outputs = model.generate(
    input_ids=inputs,
    max_new_tokens=512,
    temperature=0.2,
    do_sample=True,
)

print(tokenizer.decode(outputs[0], skip_special_tokens=True))


## 11. Compactar adapter para download opcional

Como o modelo já foi salvo no Drive, esta etapa é opcional. Ela cria um `.zip` no próprio Drive para facilitar download/backup.


In [ ]:
ZIP_PATH = OUTPUT_DIR.parent / 'llama-womens-health-triage-lora.zip'
!cd {OUTPUT_DIR.parent} && zip -r {ZIP_PATH.name} {OUTPUT_DIR.name}
print('ZIP salvo em:', ZIP_PATH)
